# 3D Cleanroom CFD + Temperature Transport + Lagrangian Particles

This notebook extends the previous 2D cleanroom model into 3D.

It includes:

```text
3D incompressible airflow
3D temperature transport
3D inertial Lagrangian particles
Gauss-Seidel pressure solver
PyVista visualization
two separate animations
```

The two animations are:

```text
1. Particle movement + physical boundary setup only
2. Temperature field + physical boundary setup only
```

This is still an educational solver, not an industrial CFD solver.

## 1. Import libraries

This notebook uses:

```text
NumPy      = numerical arrays
PyVista    = 3D visualization
tqdm       = progress bars
Path       = save file paths
```

If PyVista is not installed, run:

```bash
pip install pyvista tqdm
```

If you are using Jupyter Notebook or JupyterLab, PyVista may also need:

```bash
pip install trame
```

In [30]:
import numpy as np
import pyvista as pv

from pathlib import Path
from tqdm.notebook import tqdm

# Try to make PyVista work inside Jupyter
try:
    pv.set_jupyter_backend("static")
except Exception:
    pass

## 2. 3D physical and numerical setup

The room is now a 3D box.

```text
x direction = room length
y direction = room width
z direction = room height/depth
```

The grid shape is:

```text
field[k, j, i]
```

meaning:

```text
i = x index
j = y index
k = z index
```

So the field arrays use:

```text
[z, y, x]
```

The incompressible flow condition is:

```text
div(u) = 0
```

or in 3D:

```text
du/dx + dv/dy + dw/dz = 0
```

The pressure equation is solved using a red-black Gauss-Seidel pressure solver.

In [91]:
# -----------------------------
# 3D geometry and numerical grid
# -----------------------------
Lx = 4.0   # room length in x direction [m]
Ly = 4.0   # room width in y direction [m]
Lz = 4.0   # room height in z direction [m]

nx = 41
ny = 41
nz = 41

dx = Lx / (nx - 1)
dy = Ly / (ny - 1)
dz = Lz / (nz - 1)

x = np.linspace(0.0, Lx, nx)
y = np.linspace(0.0, Ly, ny)
z = np.linspace(0.0, Lz, nz)

# Field array shape:
# field[k, j, i] = field at z_k, y_j, x_i

# -----------------------------
# Fluid properties
# -----------------------------
rho = 1.0
nu = 0.03
U_in = 2.0

# -----------------------------
# Thermal / energy properties
# -----------------------------
rho_air = 1.2
cp_air = 1005.0
k_air = 0.0262

alpha = k_air / (rho_air * cp_air)

T_initial = 298.15
T_inlet = 308.15
T_aluminum = 298.15

# Moving body temperature: 36.5 C
T_body = 36.5 + 273.15

Q_T = 0.0

# -----------------------------
# Inlet geometry
# -----------------------------
# Inlet is on upper z face:
# z = Lz
# flow points downward in the -z direction

inlet_diameter = 0.81
inlet_radius = inlet_diameter / 2.0
inlet_area = np.pi * inlet_radius**2

inlet_center_x = Lx / 2.0
inlet_center_y = Ly / 2.0

# -----------------------------
# Outlet strip geometry
# -----------------------------
# Outlet is on lower z face:
# z = 0
#
# Outlet width:
# outlet width = inlet area * 0.17 / outlet length

outlet_floor_fraction = 0.17

outlet_strip_length_x = Lx
outlet_area = inlet_area * outlet_floor_fraction
outlet_strip_width_y = outlet_area / outlet_strip_length_x

outlet_x_min = 0.0
outlet_x_max = Lx

outlet_center_y = Ly / 2.0
outlet_y_min = outlet_center_y - outlet_strip_width_y / 2.0
outlet_y_max = outlet_center_y + outlet_strip_width_y / 2.0

# Outlet pumping amount
outlet_flow_rate = 0.162
outlet_velocity = outlet_flow_rate / outlet_area

# -----------------------------
# Floor-attached obstacle geometry
# -----------------------------
# Two fixed rectangular prism obstacles attached to the lower z face.

obstacle_length_x = Lx
obstacle_thickness_y = 1.0
obstacle_height_z = 2.0

obstacle_x_min = 0.0
obstacle_x_max = Lx

obstacle_z_min = 0.0
obstacle_z_max = obstacle_height_z

top_obstacle_center_y = 3.09
bottom_obstacle_center_y = 0.91

top_obstacle_y_min = top_obstacle_center_y - obstacle_thickness_y / 2.0
top_obstacle_y_max = top_obstacle_center_y + obstacle_thickness_y / 2.0

bottom_obstacle_y_min = bottom_obstacle_center_y - obstacle_thickness_y / 2.0
bottom_obstacle_y_max = bottom_obstacle_center_y + obstacle_thickness_y / 2.0

# -----------------------------
# Moving rectangular body
# -----------------------------
# Size:
# x length = 1.0 m
# y width  = 0.5 m
# z height = 1.75 m
#
# Motion:
# The body moves through the room only once.
# It starts moving 1 second after the simulation begins.
# It enters from the left x boundary and exits from the right x boundary.

moving_body_length_x = 1.0
moving_body_width_y = 0.5
moving_body_height_z = 1.75

moving_body_speed = 1.5
moving_body_start_time = 1.0

moving_body_center_y = Ly / 2.0
moving_body_y_min = moving_body_center_y - moving_body_width_y / 2.0
moving_body_y_max = moving_body_center_y + moving_body_width_y / 2.0

moving_body_z_min = 0.0
moving_body_z_max = moving_body_height_z

# Body starts fully outside the left boundary.
# At t = moving_body_start_time, its right face touches x = 0.
moving_body_center_x_start = -moving_body_length_x / 2.0

# Body is considered finished after its left face passes x = Lx.
moving_body_exit_time = (
    moving_body_start_time
    +
    (Lx + moving_body_length_x) / moving_body_speed
)

# When particles stuck on the moving body are released,
# they are placed slightly inside the room and allowed to fall.
moving_body_release_margin = dx
particle_fall_release_vz = -0.25

# -----------------------------
# Lagrangian particle settings
# -----------------------------
max_particles = 50
release_every = 5
particles_per_release = 1
particle_substeps = 2

particle_release_z = Lz - 0.05

rng = np.random.default_rng(1)

# Particle actual diameter distribution
# This affects particle response time, not visual marker size.
particle_diameter_options = np.array([
    0.3e-6,
    0.5e-6,
    1.0e-6
])

particle_diameter_probabilities = np.array([
    0.70,
    0.24,
    0.06
])

particle_density = 1000.0
air_dynamic_viscosity = 1.8e-5

# Gravity points from upper z face toward lower z face.
g_particle_x = 0.0
g_particle_y = 0.0
g_particle_z = -9.81

# -----------------------------
# Particle steady-state estimate settings
# -----------------------------
# Actual particle steady state:
#     all particles released AND no active/stuck moving particles remain
#
# Estimated particle steady-state time:
#     current time + estimated time for remaining active particles to stop moving
#
# This estimate is approximate and is used only to help tune nt.

particle_speed_floor = 1.0e-4       # minimum speed used to avoid division by zero [m/s]
particle_estimate_safety_factor = 1.25
particle_no_motion_required_checks = 3

# -----------------------------
# Optimized time-step setup
# -----------------------------
CFL_target = 0.20
diffusion_target = 0.25

U_ref_all = max(
    U_in,
    outlet_velocity,
    moving_body_speed
)

U_ref_x = U_ref_all
U_ref_y = U_ref_all
U_ref_z = U_ref_all

dt_cfl_x = CFL_target * dx / U_ref_x
dt_cfl_y = CFL_target * dy / U_ref_y
dt_cfl_z = CFL_target * dz / U_ref_z

inverse_grid_square_sum = (
    1.0 / dx**2
    +
    1.0 / dy**2
    +
    1.0 / dz**2
)

dt_velocity_diffusion_safe = diffusion_target / (
    nu * inverse_grid_square_sum
)

dt_thermal_diffusion_safe = diffusion_target / (
    alpha * inverse_grid_square_sum
)

dt = min(
    dt_cfl_x,
    dt_cfl_y,
    dt_cfl_z,
    dt_velocity_diffusion_safe,
    dt_thermal_diffusion_safe
)+0.03

nt = 1000
pressure_iterations = 40
frame_interval = 15

# -----------------------------
# Steady-state detection settings
# -----------------------------
steady_check_interval = 20

temperature_steady_tolerance = 1.0e-3
temperature_steady_required_checks = 5

# Particle steady state now means:
# all particles have been released and no particles are still moving.
particle_steady_required_checks = particle_no_motion_required_checks

stop_when_both_steady = True

# -----------------------------
# Final stability quantities
# -----------------------------
CFL_x = U_ref_x * dt / dx
CFL_y = U_ref_y * dt / dy
CFL_z = U_ref_z * dt / dz

velocity_diffusion_number = nu * dt * inverse_grid_square_sum
thermal_diffusion_number = alpha * dt * inverse_grid_square_sum

dt_velocity_diffusion_limit = 0.5 / (
    nu * inverse_grid_square_sum
)

dt_thermal_diffusion_limit = 0.5 / (
    alpha * inverse_grid_square_sum
)

physical_simulation_time = nt * dt

particle_tau_options = (
    particle_density * particle_diameter_options**2
    /
    (18.0 * air_dynamic_viscosity)
)

inlet_flow_rate = U_in * inlet_area

# -----------------------------
# Print setup summary
# -----------------------------
print("3D grid setup")
print(f"nx = {nx}")
print(f"ny = {ny}")
print(f"nz = {nz}")
print(f"dx = {dx:.6f} m")
print(f"dy = {dy:.6f} m")
print(f"dz = {dz:.6f} m")

print()
print("Flow direction")
print("Inlet face: upper z face, z = Lz")
print("Outlet face: lower z face, z = 0")
print("Main inlet velocity direction: -z")

print()
print("Outlet strip")
print(f"Inlet area = {inlet_area:.6f} m²")
print(f"Outlet area = inlet area * 0.17 = {outlet_area:.6f} m²")
print(f"Outlet strip length x = {outlet_strip_length_x:.3f} m")
print(f"Outlet strip width y = {outlet_strip_width_y:.6f} m")
print(f"Outlet y-range = {outlet_y_min:.6f} m to {outlet_y_max:.6f} m")
print(f"Outlet pumping flow rate = {outlet_flow_rate:.3f} m³/s")
print(f"Outlet velocity magnitude = {outlet_velocity:.3f} m/s")
print(f"Inlet flow rate from U_in = {inlet_flow_rate:.3f} m³/s")

if abs(inlet_flow_rate - outlet_flow_rate) / max(outlet_flow_rate, 1e-12) > 0.10:
    print("Warning: inlet flow rate and outlet pumping flow rate are not balanced.")

print()
print("Moving body")
print(f"Body size = {moving_body_length_x:.2f} m × {moving_body_width_y:.2f} m × {moving_body_height_z:.2f} m")
print(f"Body speed = {moving_body_speed:.2f} m/s")
print(f"Body temperature = {T_body - 273.15:.1f} °C")
print(f"Body start time = {moving_body_start_time:.3f} s")
print(f"Body exit time = {moving_body_exit_time:.3f} s")
print("Body motion: one pass only, left to right")

print()
print("Particle diameter distribution")
for d, p_prob, tau_val in zip(
    particle_diameter_options,
    particle_diameter_probabilities,
    particle_tau_options
):
    print(
        f"{100.0 * p_prob:.1f}% at {d * 1e6:.1f} µm, "
        f"estimated tau = {tau_val:.3e} s"
    )

print()
print("Particle steady-state definition")
print("Particle steady state = all particles released AND no active/stuck moving particles remain")
print(f"Particle estimate safety factor = {particle_estimate_safety_factor:.2f}")

print()
print("Optimized dt selection")
print(f"Reference speed used = {U_ref_all:.3f} m/s")
print(f"dt from x-CFL limit = {dt_cfl_x:.6f} s")
print(f"dt from y-CFL limit = {dt_cfl_y:.6f} s")
print(f"dt from z-CFL limit = {dt_cfl_z:.6f} s")
print(f"dt from velocity diffusion safe limit = {dt_velocity_diffusion_safe:.6f} s")
print(f"dt from thermal diffusion safe limit = {dt_thermal_diffusion_safe:.6f} s")
print(f"Selected dt = {dt:.6f} s")

print()
print("Final stability values")
print(f"CFL_x = {CFL_x:.4f}")
print(f"CFL_y = {CFL_y:.4f}")
print(f"CFL_z = {CFL_z:.4f}")
print(f"Velocity diffusion number = {velocity_diffusion_number:.4f}")
print(f"Thermal diffusion number = {thermal_diffusion_number:.6f}")

print()
print("Simulation setup")
print(f"nt = {nt}")
print(f"Physical simulation time = {physical_simulation_time:.3f} s")
print(f"pressure_iterations = {pressure_iterations}")
print(f"Total stored frames = {nt // frame_interval}")
print(f"Maximum particles = {max_particles}")
print(f"Particle gravity = ({g_particle_x:.2f}, {g_particle_y:.2f}, {g_particle_z:.2f}) m/s²")

3D grid setup
nx = 41
ny = 41
nz = 41
dx = 0.100000 m
dy = 0.100000 m
dz = 0.100000 m

Flow direction
Inlet face: upper z face, z = Lz
Outlet face: lower z face, z = 0
Main inlet velocity direction: -z

Outlet strip
Inlet area = 0.515300 m²
Outlet area = inlet area * 0.17 = 0.087601 m²
Outlet strip length x = 4.000 m
Outlet strip width y = 0.021900 m
Outlet y-range = 1.989050 m to 2.010950 m
Outlet pumping flow rate = 0.162 m³/s
Outlet velocity magnitude = 1.849 m/s
Inlet flow rate from U_in = 1.031 m³/s

Moving body
Body size = 1.00 m × 0.50 m × 1.75 m
Body speed = 1.50 m/s
Body temperature = 36.5 °C
Body start time = 1.000 s
Body exit time = 4.333 s
Body motion: one pass only, left to right

Particle diameter distribution
70.0% at 0.3 µm, estimated tau = 2.778e-07 s
24.0% at 0.5 µm, estimated tau = 7.716e-07 s
6.0% at 1.0 µm, estimated tau = 3.086e-06 s

Particle steady-state definition
Particle steady state = all particles released AND no active/stuck moving particles remain
Particl

## 3. 3D fields, inlet/outlet, and rectangular obstacles

The unknown fields are:

```text
u = x-direction velocity
v = y-direction velocity
w = z-direction velocity
p = pressure
T = temperature
```

The inlet is a circular disk on the left wall:

```text
x = 0
diameter = 0.81 m
center = (y = 2.0 m, z = 2.0 m)
```

The outlet is a circular disk on the right wall:

```text
x = 4.0 m
diameter = 0.34 m
center = (y = 2.0 m, z = 2.0 m)
```

The spherical/circular obstacle is removed.

Instead, there are two rectangular aluminum obstacles attached to the right wall.

Each obstacle has:

```text
x length = 2.0 m
y thickness = 1.0 m
z depth = full room depth
```

So each obstacle is a rectangular prism.

Their y-centers are:

```text
top obstacle center y = 3.09 m
bottom obstacle center y = 0.91 m
```

In [92]:
# -----------------------------
# 3D flow fields
# Shape: [z, y, x]
# -----------------------------
u_field = np.zeros((nz, ny, nx))
v_field = np.zeros((nz, ny, nx))
w_field = np.zeros((nz, ny, nx))
p = np.zeros((nz, ny, nx))

T_field = T_initial * np.ones((nz, ny, nx))

# -----------------------------
# Inlet / outlet masks on z-faces
# Face shape: [y, x]
# -----------------------------
X_face, Y_face = np.meshgrid(x, y)

# Inlet is a circular disk on upper z face, z = Lz
inlet = (
    (X_face - inlet_center_x)**2
    +
    (Y_face - inlet_center_y)**2
) <= inlet_radius**2

# Outlet is a long rectangular strip on lower z face, z = 0
outlet = (
    (X_face >= outlet_x_min)
    &
    (X_face <= outlet_x_max)
    &
    (Y_face >= outlet_y_min)
    &
    (Y_face <= outlet_y_max)
)

# -----------------------------
# Fixed floor-attached rectangular prism obstacles
# -----------------------------
X3 = x[None, None, :]
Y3 = y[None, :, None]
Z3 = z[:, None, None]

top_obstacle = (
    (X3 >= obstacle_x_min)
    &
    (X3 <= obstacle_x_max)
    &
    (Y3 >= top_obstacle_y_min)
    &
    (Y3 <= top_obstacle_y_max)
    &
    (Z3 >= obstacle_z_min)
    &
    (Z3 <= obstacle_z_max)
)

bottom_obstacle = (
    (X3 >= obstacle_x_min)
    &
    (X3 <= obstacle_x_max)
    &
    (Y3 >= bottom_obstacle_y_min)
    &
    (Y3 <= bottom_obstacle_y_max)
    &
    (Z3 >= obstacle_z_min)
    &
    (Z3 <= obstacle_z_max)
)

fixed_obstacle = top_obstacle | bottom_obstacle

# -----------------------------
# Moving body functions
# -----------------------------
def get_moving_body_state(time_value):
    """
    Return moving body bounds, x velocity, and active state.

    New motion:
        t < 1 s:
            body is outside the room and inactive

        t >= 1 s:
            body moves once from left to right

        after it exits:
            body becomes inactive again
    """

    if time_value < moving_body_start_time:
        return None, 0.0, False

    elapsed_time = time_value - moving_body_start_time

    center_x = (
        moving_body_center_x_start
        +
        moving_body_speed * elapsed_time
    )

    x_min_body = center_x - moving_body_length_x / 2.0
    x_max_body = center_x + moving_body_length_x / 2.0

    # Active while any part of the body intersects the room.
    body_active = (
        x_max_body >= 0.0
        and
        x_min_body <= Lx
    )

    if not body_active:
        return None, 0.0, False

    bounds = (
        x_min_body,
        x_max_body,
        moving_body_y_min,
        moving_body_y_max,
        moving_body_z_min,
        moving_body_z_max
    )

    return bounds, moving_body_speed, True


def get_moving_body_mask(time_value):
    """
    Return boolean mask for moving rectangular body.
    """

    body_bounds, body_velocity_x, body_active = get_moving_body_state(time_value)

    if not body_active:
        body_mask = np.zeros((nz, ny, nx), dtype=bool)
        return body_mask, None, 0.0, False

    x_min_body, x_max_body, y_min_body, y_max_body, z_min_body, z_max_body = body_bounds

    body_mask = (
        (X3 >= x_min_body)
        &
        (X3 <= x_max_body)
        &
        (Y3 >= y_min_body)
        &
        (Y3 <= y_max_body)
        &
        (Z3 >= z_min_body)
        &
        (Z3 <= z_max_body)
    )

    return body_mask, body_bounds, body_velocity_x, body_active


# Initial moving body
initial_body_mask, initial_body_bounds, initial_body_velocity_x, initial_body_active = get_moving_body_mask(0.0)

solid_mask = fixed_obstacle | initial_body_mask
fluid = ~solid_mask

T_field[fixed_obstacle] = T_aluminum

if initial_body_active:
    T_field[initial_body_mask] = T_body

# -----------------------------
# Red-black patterns for Gauss-Seidel pressure solver
# Actual fluid mask changes because the body moves.
# -----------------------------
K_i, J_i, I_i = np.indices((nz - 2, ny - 2, nx - 2))

red_pattern = ((I_i + J_i + K_i) % 2 == 0)
black_pattern = ((I_i + J_i + K_i) % 2 == 1)

print("Inlet face grid points:", np.sum(inlet))
print("Outlet face grid points:", np.sum(outlet))
print("Fixed obstacle grid points:", np.sum(fixed_obstacle))
print("Initial moving body grid points:", np.sum(initial_body_mask))
print("Initial moving body active:", initial_body_active)

print()
print("Outlet")
print(f"Outlet area = {outlet_area:.6f} m²")
print(f"Outlet width = {outlet_strip_width_y:.6f} m")
print(f"Outlet pumping flow rate = {outlet_flow_rate:.3f} m³/s")
print(f"Outlet velocity = {outlet_velocity:.3f} m/s downward")

print()
print("Moving body")
print(f"Start time = {moving_body_start_time:.3f} s")
print(f"Exit time = {moving_body_exit_time:.3f} s")
print("Path = one pass from left boundary to right boundary")

Inlet face grid points: 49
Outlet face grid points: 41
Fixed obstacle grid points: 17220
Initial moving body grid points: 0
Initial moving body active: False

Outlet
Outlet area = 0.087601 m²
Outlet width = 0.021900 m
Outlet pumping flow rate = 0.162 m³/s
Outlet velocity = 1.849 m/s downward

Moving body
Start time = 1.000 s
Exit time = 4.333 s
Path = one pass from left boundary to right boundary


## 4. Boundary conditions

Velocity boundary conditions:

```text
Room walls: no-slip
Inlet: fixed x-velocity
Outlet: zero-gradient velocity
Obstacles: no-slip solid
```

Pressure boundary conditions:

```text
Room walls: zero normal pressure gradient
Outlet: reference pressure p = 0
Obstacles: approximate zero normal pressure gradient
```

Temperature boundary conditions:

```text
Room walls: zero-flux temperature
Inlet: fixed temperature
Outlet: zero-gradient temperature
Obstacles: constant aluminum temperature
```

In [93]:
def average_inside_solid_from_neighbors(field, mask):
    """
    Fill solid values using neighbor averages.
    This is only a simple educational approximation.
    """

    avg = np.zeros_like(field)
    count = np.zeros_like(field)

    avg[1:, :, :] += field[:-1, :, :]
    count[1:, :, :] += 1.0

    avg[:-1, :, :] += field[1:, :, :]
    count[:-1, :, :] += 1.0

    avg[:, 1:, :] += field[:, :-1, :]
    count[:, 1:, :] += 1.0

    avg[:, :-1, :] += field[:, 1:, :]
    count[:, :-1, :] += 1.0

    avg[:, :, 1:] += field[:, :, :-1]
    count[:, :, 1:] += 1.0

    avg[:, :, :-1] += field[:, :, 1:]
    count[:, :, :-1] += 1.0

    neighbor_average = avg / np.maximum(count, 1.0)
    field[mask] = neighbor_average[mask]

    return field


def apply_velocity_bcs(u, v, w, solid_mask, body_mask, body_velocity_x):
    """
    3D velocity boundary conditions.

    Inlet:
        upper z face, z = Lz
        velocity points downward, w = -U_in

    Outlet:
        lower z face, z = 0
        prescribed pumping velocity from outlet flow rate

    Moving body:
        u = body_velocity_x, v = 0, w = 0
    """

    # x walls
    u[:, :, 0] = 0.0
    v[:, :, 0] = 0.0
    w[:, :, 0] = 0.0

    u[:, :, -1] = 0.0
    v[:, :, -1] = 0.0
    w[:, :, -1] = 0.0

    # y walls
    u[:, 0, :] = 0.0
    v[:, 0, :] = 0.0
    w[:, 0, :] = 0.0

    u[:, -1, :] = 0.0
    v[:, -1, :] = 0.0
    w[:, -1, :] = 0.0

    # lower z wall
    u[0, :, :] = 0.0
    v[0, :, :] = 0.0
    w[0, :, :] = 0.0

    # upper z wall
    u[-1, :, :] = 0.0
    v[-1, :, :] = 0.0
    w[-1, :, :] = 0.0

    # Inlet on upper z face
    upper_u = u[-1, :, :].copy()
    upper_v = v[-1, :, :].copy()
    upper_w = w[-1, :, :].copy()

    upper_u[inlet] = 0.0
    upper_v[inlet] = 0.0
    upper_w[inlet] = -U_in

    u[-1, :, :] = upper_u
    v[-1, :, :] = upper_v
    w[-1, :, :] = upper_w

    # Outlet pumping on lower z face
    lower_u = u[0, :, :].copy()
    lower_v = v[0, :, :].copy()
    lower_w = w[0, :, :].copy()

    lower_u[outlet] = 0.0
    lower_v[outlet] = 0.0
    lower_w[outlet] = -outlet_velocity

    u[0, :, :] = lower_u
    v[0, :, :] = lower_v
    w[0, :, :] = lower_w

    # Fixed obstacles
    u[fixed_obstacle] = 0.0
    v[fixed_obstacle] = 0.0
    w[fixed_obstacle] = 0.0

    # Moving body
    u[body_mask] = body_velocity_x
    v[body_mask] = 0.0
    w[body_mask] = 0.0

    return u, v, w


def apply_pressure_bcs(p_field, solid_mask):
    """
    3D pressure boundary conditions.
    """

    # x walls
    p_field[:, :, 0] = p_field[:, :, 1]
    p_field[:, :, -1] = p_field[:, :, -2]

    # y walls
    p_field[:, 0, :] = p_field[:, 1, :]
    p_field[:, -1, :] = p_field[:, -2, :]

    # z walls
    p_field[0, :, :] = p_field[1, :, :]
    p_field[-1, :, :] = p_field[-2, :, :]

    # Outlet reference pressure on lower z face
    lower_p = p_field[0, :, :].copy()
    lower_p[outlet] = 0.0
    p_field[0, :, :] = lower_p

    p_field = average_inside_solid_from_neighbors(p_field, solid_mask)

    return p_field


def apply_temperature_bcs(T, solid_mask, body_mask):
    """
    3D temperature boundary conditions.

    Inlet:
        upper z face, fixed T = T_inlet

    Outlet:
        lower z face outlet strip, zero-gradient temperature

    Fixed obstacles:
        T = T_aluminum

    Moving body:
        T = T_body
    """

    # x wall zero-flux
    T[:, :, 0] = T[:, :, 1]
    T[:, :, -1] = T[:, :, -2]

    # y wall zero-flux
    T[:, 0, :] = T[:, 1, :]
    T[:, -1, :] = T[:, -2, :]

    # z wall zero-flux
    T[0, :, :] = T[1, :, :]
    T[-1, :, :] = T[-2, :, :]

    # Fixed inlet temperature on upper z face
    upper_T = T[-1, :, :].copy()
    upper_T[inlet] = T_inlet
    T[-1, :, :] = upper_T

    # Outlet zero-gradient on lower z face
    lower_T = T[0, :, :].copy()
    lower_T[outlet] = T[1, :, :][outlet]
    T[0, :, :] = lower_T

    # Fixed obstacles
    T[fixed_obstacle] = T_aluminum

    # Moving body
    T[body_mask] = T_body

    return T

## 5. Pressure Poisson equation with Gauss-Seidel solver

The incompressibility condition is:

```text
div(u) = 0
```

In the projection method, the solver first predicts a temporary velocity field:

```text
u*
v*
w*
```

Then pressure is solved from:

```text
∇²p = rho / dt · div(u*)
```

In 3D:

```text
∇²p = d²p/dx² + d²p/dy² + d²p/dz²
```

This notebook uses a red-black Gauss-Seidel method instead of the older Jacobi-style pressure solver.

Gauss-Seidel usually converges faster because it immediately uses updated pressure values.

In [94]:
def build_pressure_rhs(u_star, v_star, w_star, solid_mask):
    """
    Build RHS for 3D pressure Poisson equation.

    b = rho/dt * div(u*)
    """

    b = np.zeros_like(p)

    b[1:-1, 1:-1, 1:-1] = (rho / dt) * (
        (u_star[1:-1, 1:-1, 2:] - u_star[1:-1, 1:-1, :-2]) / (2.0 * dx)
        +
        (v_star[1:-1, 2:, 1:-1] - v_star[1:-1, :-2, 1:-1]) / (2.0 * dy)
        +
        (w_star[2:, 1:-1, 1:-1] - w_star[:-2, 1:-1, 1:-1]) / (2.0 * dz)
    )

    b[solid_mask] = 0.0

    return b


def pressure_candidate_3d(p_field, b):
    """
    Candidate pressure update for 3D Poisson equation.
    """

    dx2 = dx**2
    dy2 = dy**2
    dz2 = dz**2

    denominator = 2.0 * (
        dy2 * dz2
        +
        dx2 * dz2
        +
        dx2 * dy2
    )

    candidate = (
        (p_field[1:-1, 1:-1, 2:] + p_field[1:-1, 1:-1, :-2]) * dy2 * dz2
        +
        (p_field[1:-1, 2:, 1:-1] + p_field[1:-1, :-2, 1:-1]) * dx2 * dz2
        +
        (p_field[2:, 1:-1, 1:-1] + p_field[:-2, 1:-1, 1:-1]) * dx2 * dy2
        -
        b[1:-1, 1:-1, 1:-1] * dx2 * dy2 * dz2
    ) / denominator

    return candidate


def solve_pressure_gauss_seidel(p_field, b, solid_mask):
    """
    Red-black Gauss-Seidel pressure solver.
    The moving body changes the fluid mask every time step.
    """

    fluid_interior_mask = (~solid_mask)[1:-1, 1:-1, 1:-1]

    red_mask = red_pattern & fluid_interior_mask
    black_mask = black_pattern & fluid_interior_mask

    for _ in range(pressure_iterations):

        # Red update
        candidate = pressure_candidate_3d(p_field, b)
        center = p_field[1:-1, 1:-1, 1:-1]
        center[red_mask] = candidate[red_mask]
        p_field[1:-1, 1:-1, 1:-1] = center

        p_field = apply_pressure_bcs(p_field, solid_mask)

        # Black update
        candidate = pressure_candidate_3d(p_field, b)
        center = p_field[1:-1, 1:-1, 1:-1]
        center[black_mask] = candidate[black_mask]
        p_field[1:-1, 1:-1, 1:-1] = center

        p_field = apply_pressure_bcs(p_field, solid_mask)

    return p_field

## 6. 3D incompressible velocity-pressure step

Each time step does:

```text
1. Predict velocity without pressure
2. Solve pressure using Gauss-Seidel
3. Correct velocity using pressure gradient
4. Re-apply boundary conditions
```

The momentum equation is solved in simplified form:

```text
du/dt + u du/dx + v du/dy + w du/dz = -1/rho dp/dx + nu ∇²u
```

and similarly for:

```text
v velocity
w velocity
```

In [95]:
def step_velocity_pressure_3d(
    u,
    v,
    w,
    p_field,
    solid_mask,
    body_mask,
    body_velocity_x
):
    """
    Advance 3D incompressible velocity-pressure field by one time step.
    """

    u = u.copy()
    v = v.copy()
    w = w.copy()

    un = u.copy()
    vn = v.copy()
    wn = w.copy()

    u_star = un.copy()
    v_star = vn.copy()
    w_star = wn.copy()

    fluid_interior = (~solid_mask)[1:-1, 1:-1, 1:-1]

    # -----------------------------
    # Derivatives
    # -----------------------------
    du_dx = (un[1:-1, 1:-1, 2:] - un[1:-1, 1:-1, :-2]) / (2.0 * dx)
    du_dy = (un[1:-1, 2:, 1:-1] - un[1:-1, :-2, 1:-1]) / (2.0 * dy)
    du_dz = (un[2:, 1:-1, 1:-1] - un[:-2, 1:-1, 1:-1]) / (2.0 * dz)

    dv_dx = (vn[1:-1, 1:-1, 2:] - vn[1:-1, 1:-1, :-2]) / (2.0 * dx)
    dv_dy = (vn[1:-1, 2:, 1:-1] - vn[1:-1, :-2, 1:-1]) / (2.0 * dy)
    dv_dz = (vn[2:, 1:-1, 1:-1] - vn[:-2, 1:-1, 1:-1]) / (2.0 * dz)

    dw_dx = (wn[1:-1, 1:-1, 2:] - wn[1:-1, 1:-1, :-2]) / (2.0 * dx)
    dw_dy = (wn[1:-1, 2:, 1:-1] - wn[1:-1, :-2, 1:-1]) / (2.0 * dy)
    dw_dz = (wn[2:, 1:-1, 1:-1] - wn[:-2, 1:-1, 1:-1]) / (2.0 * dz)

    # -----------------------------
    # Laplacians
    # -----------------------------
    lap_u = (
        (un[1:-1, 1:-1, 2:] - 2.0 * un[1:-1, 1:-1, 1:-1] + un[1:-1, 1:-1, :-2]) / dx**2
        +
        (un[1:-1, 2:, 1:-1] - 2.0 * un[1:-1, 1:-1, 1:-1] + un[1:-1, :-2, 1:-1]) / dy**2
        +
        (un[2:, 1:-1, 1:-1] - 2.0 * un[1:-1, 1:-1, 1:-1] + un[:-2, 1:-1, 1:-1]) / dz**2
    )

    lap_v = (
        (vn[1:-1, 1:-1, 2:] - 2.0 * vn[1:-1, 1:-1, 1:-1] + vn[1:-1, 1:-1, :-2]) / dx**2
        +
        (vn[1:-1, 2:, 1:-1] - 2.0 * vn[1:-1, 1:-1, 1:-1] + vn[1:-1, :-2, 1:-1]) / dy**2
        +
        (vn[2:, 1:-1, 1:-1] - 2.0 * vn[1:-1, 1:-1, 1:-1] + vn[:-2, 1:-1, 1:-1]) / dz**2
    )

    lap_w = (
        (wn[1:-1, 1:-1, 2:] - 2.0 * wn[1:-1, 1:-1, 1:-1] + wn[1:-1, 1:-1, :-2]) / dx**2
        +
        (wn[1:-1, 2:, 1:-1] - 2.0 * wn[1:-1, 1:-1, 1:-1] + wn[1:-1, :-2, 1:-1]) / dy**2
        +
        (wn[2:, 1:-1, 1:-1] - 2.0 * wn[1:-1, 1:-1, 1:-1] + wn[:-2, 1:-1, 1:-1]) / dz**2
    )

    # -----------------------------
    # Predict velocity
    # -----------------------------
    predicted_u = un[1:-1, 1:-1, 1:-1] + dt * (
        -(un[1:-1, 1:-1, 1:-1] * du_dx
          + vn[1:-1, 1:-1, 1:-1] * du_dy
          + wn[1:-1, 1:-1, 1:-1] * du_dz)
        +
        nu * lap_u
    )

    predicted_v = vn[1:-1, 1:-1, 1:-1] + dt * (
        -(un[1:-1, 1:-1, 1:-1] * dv_dx
          + vn[1:-1, 1:-1, 1:-1] * dv_dy
          + wn[1:-1, 1:-1, 1:-1] * dv_dz)
        +
        nu * lap_v
    )

    predicted_w = wn[1:-1, 1:-1, 1:-1] + dt * (
        -(un[1:-1, 1:-1, 1:-1] * dw_dx
          + vn[1:-1, 1:-1, 1:-1] * dw_dy
          + wn[1:-1, 1:-1, 1:-1] * dw_dz)
        +
        nu * lap_w
    )

    u_star[1:-1, 1:-1, 1:-1] = np.where(fluid_interior, predicted_u, 0.0)
    v_star[1:-1, 1:-1, 1:-1] = np.where(fluid_interior, predicted_v, 0.0)
    w_star[1:-1, 1:-1, 1:-1] = np.where(fluid_interior, predicted_w, 0.0)

    u_star, v_star, w_star = apply_velocity_bcs(
        u_star,
        v_star,
        w_star,
        solid_mask,
        body_mask,
        body_velocity_x
    )

    # -----------------------------
    # Pressure solve
    # -----------------------------
    b = build_pressure_rhs(u_star, v_star, w_star, solid_mask)
    p_field = solve_pressure_gauss_seidel(p_field, b, solid_mask)

    # -----------------------------
    # Correct velocity
    # -----------------------------
    u_new = u_star.copy()
    v_new = v_star.copy()
    w_new = w_star.copy()

    corrected_u = u_star[1:-1, 1:-1, 1:-1] - (dt / rho) * (
        (p_field[1:-1, 1:-1, 2:] - p_field[1:-1, 1:-1, :-2]) / (2.0 * dx)
    )

    corrected_v = v_star[1:-1, 1:-1, 1:-1] - (dt / rho) * (
        (p_field[1:-1, 2:, 1:-1] - p_field[1:-1, :-2, 1:-1]) / (2.0 * dy)
    )

    corrected_w = w_star[1:-1, 1:-1, 1:-1] - (dt / rho) * (
        (p_field[2:, 1:-1, 1:-1] - p_field[:-2, 1:-1, 1:-1]) / (2.0 * dz)
    )

    u_new[1:-1, 1:-1, 1:-1] = np.where(fluid_interior, corrected_u, 0.0)
    v_new[1:-1, 1:-1, 1:-1] = np.where(fluid_interior, corrected_v, 0.0)
    w_new[1:-1, 1:-1, 1:-1] = np.where(fluid_interior, corrected_w, 0.0)

    u_new, v_new, w_new = apply_velocity_bcs(
        u_new,
        v_new,
        w_new,
        solid_mask,
        body_mask,
        body_velocity_x
    )

    return u_new, v_new, w_new, p_field

## 7. 3D temperature transport equation

The temperature-based energy equation is:

```text
rho cp (dT/dt + u dT/dx + v dT/dy + w dT/dz) = div(k grad(T)) + Q
```

Assuming constant thermal conductivity:

```text
grad(k) = 0
```

so:

```text
div(k grad(T)) = k ∇²T
```

After dividing by `rho cp`:

```text
dT/dt + u dT/dx + v dT/dy + w dT/dz = alpha ∇²T + Q/(rho cp)
```

where:

```text
alpha = k / (rho cp)
```

The flow is incompressible, so the conservative and advective forms are equivalent.

In [96]:
def step_temperature_3d(T, u, v, w, solid_mask, body_mask):
    """
    Advance 3D temperature field by one step.

    dT/dt + u dT/dx + v dT/dy + w dT/dz
    =
    alpha ∇²T + Q/(rho cp)
    """

    T = T.copy()
    Tn = T.copy()

    fluid_interior = (~solid_mask)[1:-1, 1:-1, 1:-1]

    # Upwind derivatives
    dTdx_backward = (Tn[1:-1, 1:-1, 1:-1] - Tn[1:-1, 1:-1, :-2]) / dx
    dTdx_forward = (Tn[1:-1, 1:-1, 2:] - Tn[1:-1, 1:-1, 1:-1]) / dx

    dTdy_backward = (Tn[1:-1, 1:-1, 1:-1] - Tn[1:-1, :-2, 1:-1]) / dy
    dTdy_forward = (Tn[1:-1, 2:, 1:-1] - Tn[1:-1, 1:-1, 1:-1]) / dy

    dTdz_backward = (Tn[1:-1, 1:-1, 1:-1] - Tn[:-2, 1:-1, 1:-1]) / dz
    dTdz_forward = (Tn[2:, 1:-1, 1:-1] - Tn[1:-1, 1:-1, 1:-1]) / dz

    dT_dx = np.where(u[1:-1, 1:-1, 1:-1] >= 0.0, dTdx_backward, dTdx_forward)
    dT_dy = np.where(v[1:-1, 1:-1, 1:-1] >= 0.0, dTdy_backward, dTdy_forward)
    dT_dz = np.where(w[1:-1, 1:-1, 1:-1] >= 0.0, dTdz_backward, dTdz_forward)

    lap_T = (
        (Tn[1:-1, 1:-1, 2:] - 2.0 * Tn[1:-1, 1:-1, 1:-1] + Tn[1:-1, 1:-1, :-2]) / dx**2
        +
        (Tn[1:-1, 2:, 1:-1] - 2.0 * Tn[1:-1, 1:-1, 1:-1] + Tn[1:-1, :-2, 1:-1]) / dy**2
        +
        (Tn[2:, 1:-1, 1:-1] - 2.0 * Tn[1:-1, 1:-1, 1:-1] + Tn[:-2, 1:-1, 1:-1]) / dz**2
    )

    T_candidate = Tn[1:-1, 1:-1, 1:-1] + dt * (
        -(u[1:-1, 1:-1, 1:-1] * dT_dx
          + v[1:-1, 1:-1, 1:-1] * dT_dy
          + w[1:-1, 1:-1, 1:-1] * dT_dz)
        +
        alpha * lap_T
        +
        Q_T / (rho_air * cp_air)
    )

    T[1:-1, 1:-1, 1:-1] = np.where(
        fluid_interior,
        T_candidate,
        T[1:-1, 1:-1, 1:-1]
    )

    T = apply_temperature_bcs(T, solid_mask, body_mask)

    return T

## 8. 3D Lagrangian particles

Particles are now released from the upper z face and move downward toward the lower z face.

Particle position equations:

```text
dxp/dt = vpx
dyp/dt = vpy
dzp/dt = vpz
```

Particle velocity equations:

```text
dvpx/dt = (u_air - vpx) / tau_p + gx
dvpy/dt = (v_air - vpy) / tau_p + gy
dvpz/dt = (w_air - vpz) / tau_p + gz
```

where:

```text
xp, yp, zp            = particle position
vpx, vpy, vpz         = particle velocity
u_air, v_air, w_air   = interpolated air velocity
tau_p                 = particle response time
gx, gy, gz            = particle acceleration
```

In this simulation, the particle acceleration follows the new flow direction:

```text
gx = 0
gy = 0
gz = -9.81 m/s²
```

So particles are accelerated from the upper z face toward the lower z face.

Particles can be:

```text
0 = unreleased
1 = active
2 = escaped through outlet strip
3 = deposited on wall
4 = deposited on obstacle
```

In [97]:
def compute_particle_response_time(particle_diameter):
    """
    Compute particle response time using simplified Stokes drag:

        tau_p = rho_p d_p^2 / (18 mu)

    This uses the actual particle diameter.
    """

    tau_p = (
        particle_density
        *
        particle_diameter**2
        /
        (18.0 * air_dynamic_viscosity)
    )

    return np.maximum(tau_p, 1.0e-12)


def trilinear_interpolate_velocity(xp, yp, zp, u, v, w):
    """
    Interpolate 3D velocity to particle locations.
    """

    xi = xp / dx
    yi = yp / dy
    zi = zp / dz

    i0 = np.floor(xi).astype(int)
    j0 = np.floor(yi).astype(int)
    k0 = np.floor(zi).astype(int)

    i0 = np.clip(i0, 0, nx - 2)
    j0 = np.clip(j0, 0, ny - 2)
    k0 = np.clip(k0, 0, nz - 2)

    i1 = i0 + 1
    j1 = j0 + 1
    k1 = k0 + 1

    sx = xi - i0
    sy = yi - j0
    sz = zi - k0

    def interp(field):
        c000 = field[k0, j0, i0]
        c100 = field[k0, j0, i1]
        c010 = field[k0, j1, i0]
        c110 = field[k0, j1, i1]

        c001 = field[k1, j0, i0]
        c101 = field[k1, j0, i1]
        c011 = field[k1, j1, i0]
        c111 = field[k1, j1, i1]

        return (
            c000 * (1 - sx) * (1 - sy) * (1 - sz)
            + c100 * sx * (1 - sy) * (1 - sz)
            + c010 * (1 - sx) * sy * (1 - sz)
            + c110 * sx * sy * (1 - sz)
            + c001 * (1 - sx) * (1 - sy) * sz
            + c101 * sx * (1 - sy) * sz
            + c011 * (1 - sx) * sy * sz
            + c111 * sx * sy * sz
        )

    up = interp(u)
    vp = interp(v)
    wp = interp(w)

    return up, vp, wp


def inside_fixed_obstacle_particles_3d(xp, yp, zp):
    """
    Check whether particles are inside either fixed floor-attached obstacle.
    """

    inside_top = (
        (xp >= obstacle_x_min)
        &
        (xp <= obstacle_x_max)
        &
        (yp >= top_obstacle_y_min)
        &
        (yp <= top_obstacle_y_max)
        &
        (zp >= obstacle_z_min)
        &
        (zp <= obstacle_z_max)
    )

    inside_bottom = (
        (xp >= obstacle_x_min)
        &
        (xp <= obstacle_x_max)
        &
        (yp >= bottom_obstacle_y_min)
        &
        (yp <= bottom_obstacle_y_max)
        &
        (zp >= obstacle_z_min)
        &
        (zp <= obstacle_z_max)
    )

    return inside_top | inside_bottom


def inside_moving_body_particles_3d(xp, yp, zp, body_bounds, body_active):
    """
    Check whether particles are inside the moving rectangular body.
    """

    if not body_active or body_bounds is None:
        return np.zeros_like(xp, dtype=bool)

    x_min_body, x_max_body, y_min_body, y_max_body, z_min_body, z_max_body = body_bounds

    inside_body = (
        (xp >= x_min_body)
        &
        (xp <= x_max_body)
        &
        (yp >= y_min_body)
        &
        (yp <= y_max_body)
        &
        (zp >= z_min_body)
        &
        (zp <= z_max_body)
    )

    return inside_body


def release_particles_3d(
    particle_x,
    particle_y,
    particle_z,
    particle_vx,
    particle_vy,
    particle_vz,
    particle_diameter,
    particle_tau,
    particle_body_offset_x,
    particle_body_offset_y,
    particle_body_offset_z,
    particle_status,
    next_particle_id
):
    """
    Release particles uniformly over the circular inlet disk on the upper z face.

    Actual particle diameter is sampled from:

        70% at 0.3 micron
        24% at 0.5 micron
        6%  at 1.0 micron

    Visual marker size is not changed by this diameter.
    """

    for _ in range(particles_per_release):
        if next_particle_id >= max_particles:
            break

        r = inlet_radius * np.sqrt(rng.random())
        theta = 2.0 * np.pi * rng.random()

        particle_x[next_particle_id] = inlet_center_x + r * np.cos(theta)
        particle_y[next_particle_id] = inlet_center_y + r * np.sin(theta)
        particle_z[next_particle_id] = particle_release_z

        particle_vx[next_particle_id] = 0.0
        particle_vy[next_particle_id] = 0.0
        particle_vz[next_particle_id] = -U_in

        selected_diameter = rng.choice(
            particle_diameter_options,
            p=particle_diameter_probabilities
        )

        particle_diameter[next_particle_id] = selected_diameter
        particle_tau[next_particle_id] = compute_particle_response_time(selected_diameter)

        particle_body_offset_x[next_particle_id] = np.nan
        particle_body_offset_y[next_particle_id] = np.nan
        particle_body_offset_z[next_particle_id] = np.nan

        particle_status[next_particle_id] = 1
        next_particle_id += 1

    return (
        particle_x,
        particle_y,
        particle_z,
        particle_vx,
        particle_vy,
        particle_vz,
        particle_diameter,
        particle_tau,
        particle_body_offset_x,
        particle_body_offset_y,
        particle_body_offset_z,
        particle_status,
        next_particle_id
    )


def release_stuck_particles_when_body_leaves(
    particle_x,
    particle_y,
    particle_z,
    particle_vx,
    particle_vy,
    particle_vz,
    particle_status
):
    """
    If particles are stuck on the moving body and the body leaves the room,
    release those particles back into the airflow and let them fall downward.
    """

    stuck = particle_status == 5

    if np.any(stuck):
        particle_x[stuck] = np.clip(
            particle_x[stuck],
            moving_body_release_margin,
            Lx - moving_body_release_margin
        )

        particle_y[stuck] = np.clip(
            particle_y[stuck],
            moving_body_release_margin,
            Ly - moving_body_release_margin
        )

        particle_z[stuck] = np.clip(
            particle_z[stuck],
            moving_body_release_margin,
            Lz - moving_body_release_margin
        )

        particle_vx[stuck] = 0.0
        particle_vy[stuck] = 0.0
        particle_vz[stuck] = particle_fall_release_vz

        particle_status[stuck] = 1

    return (
        particle_x,
        particle_y,
        particle_z,
        particle_vx,
        particle_vy,
        particle_vz,
        particle_status
    )


def step_particles_3d(
    particle_x,
    particle_y,
    particle_z,
    particle_vx,
    particle_vy,
    particle_vz,
    particle_diameter,
    particle_tau,
    particle_body_offset_x,
    particle_body_offset_y,
    particle_body_offset_z,
    particle_status,
    u,
    v,
    w,
    body_bounds,
    body_velocity_x,
    body_active
):
    """
    3D inertial Lagrangian particle update.

    Particle status:
        0 = unreleased
        1 = active
        2 = escaped
        3 = wall deposited
        4 = fixed obstacle deposited
        5 = stuck on moving body

    New moving-body behavior:
        If a particle hits the moving body, it sticks to it.
        When the body exits the room, the stuck particle is released
        and falls downward.
    """

    dtp = dt / particle_substeps

    for _ in range(particle_substeps):

        # -----------------------------
        # Move stuck particles with the moving body
        # -----------------------------
        stuck = particle_status == 5

        if np.any(stuck):
            if body_active and body_bounds is not None:
                x_min_body, x_max_body, y_min_body, y_max_body, z_min_body, z_max_body = body_bounds

                particle_x[stuck] = x_min_body + particle_body_offset_x[stuck]
                particle_y[stuck] = y_min_body + particle_body_offset_y[stuck]
                particle_z[stuck] = z_min_body + particle_body_offset_z[stuck]

                particle_vx[stuck] = body_velocity_x
                particle_vy[stuck] = 0.0
                particle_vz[stuck] = 0.0

            else:
                (
                    particle_x,
                    particle_y,
                    particle_z,
                    particle_vx,
                    particle_vy,
                    particle_vz,
                    particle_status
                ) = release_stuck_particles_when_body_leaves(
                    particle_x,
                    particle_y,
                    particle_z,
                    particle_vx,
                    particle_vy,
                    particle_vz,
                    particle_status
                )

        # -----------------------------
        # Move active particles
        # -----------------------------
        active = particle_status == 1

        if not np.any(active):
            continue

        xp = particle_x[active]
        yp = particle_y[active]
        zp = particle_z[active]

        vpx = particle_vx[active]
        vpy = particle_vy[active]
        vpz = particle_vz[active]

        taup = particle_tau[active]

        up, vp, wp = trilinear_interpolate_velocity(xp, yp, zp, u, v, w)

        # Stable exponential drag update
        exp_factor = np.exp(-dtp / taup)

        terminal_vx = up + taup * g_particle_x
        terminal_vy = vp + taup * g_particle_y
        terminal_vz = wp + taup * g_particle_z

        vpx_new = terminal_vx + (vpx - terminal_vx) * exp_factor
        vpy_new = terminal_vy + (vpy - terminal_vy) * exp_factor
        vpz_new = terminal_vz + (vpz - terminal_vz) * exp_factor

        xp_new = xp + vpx_new * dtp
        yp_new = yp + vpy_new * dtp
        zp_new = zp + vpz_new * dtp

        escaped = (
            (zp_new <= 0.0)
            &
            (xp_new >= outlet_x_min)
            &
            (xp_new <= outlet_x_max)
            &
            (yp_new >= outlet_y_min)
            &
            (yp_new <= outlet_y_max)
        )

        wall_deposit = (
            (xp_new <= 0.0)
            |
            (xp_new >= Lx)
            |
            (yp_new <= 0.0)
            |
            (yp_new >= Ly)
            |
            (zp_new >= Lz)
            |
            (
                (zp_new <= 0.0)
                &
                ~escaped
            )
        )

        fixed_obstacle_deposit = inside_fixed_obstacle_particles_3d(
            xp_new,
            yp_new,
            zp_new
        )

        moving_body_contact = inside_moving_body_particles_3d(
            xp_new,
            yp_new,
            zp_new,
            body_bounds,
            body_active
        )

        particle_x[active] = xp_new
        particle_y[active] = yp_new
        particle_z[active] = zp_new

        particle_vx[active] = vpx_new
        particle_vy[active] = vpy_new
        particle_vz[active] = vpz_new

        active_indices = np.where(active)[0]

        escaped_indices = active_indices[escaped]
        wall_indices = active_indices[wall_deposit]
        fixed_obstacle_indices = active_indices[fixed_obstacle_deposit]
        moving_body_indices = active_indices[moving_body_contact]

        particle_status[escaped_indices] = 2
        particle_status[wall_indices] = 3
        particle_status[fixed_obstacle_indices] = 4

        # Stuck-on-body status is separate from fixed obstacle deposition.
        # Store particle offset relative to the moving body minimum corner.
        if body_active and body_bounds is not None and len(moving_body_indices) > 0:
            x_min_body, x_max_body, y_min_body, y_max_body, z_min_body, z_max_body = body_bounds

            particle_status[moving_body_indices] = 5

            particle_body_offset_x[moving_body_indices] = (
                particle_x[moving_body_indices] - x_min_body
            )

            particle_body_offset_y[moving_body_indices] = (
                particle_y[moving_body_indices] - y_min_body
            )

            particle_body_offset_z[moving_body_indices] = (
                particle_z[moving_body_indices] - z_min_body
            )

    return (
        particle_x,
        particle_y,
        particle_z,
        particle_vx,
        particle_vy,
        particle_vz,
        particle_diameter,
        particle_tau,
        particle_body_offset_x,
        particle_body_offset_y,
        particle_body_offset_z,
        particle_status
    )

## 9. Run the 3D simulation

This cell runs the full 3D simulation:

```text
1. Solve 3D incompressible airflow
2. Solve 3D temperature transport
3. Release particles at the inlet
4. Move particles through the 3D airflow
5. Store temperature frames and particle frames
```

3D is much heavier than 2D.

For the first test, you can temporarily use:

```python
nt = 100
```

After confirming the code works, change back to:

```python
nt = 1800
```

In [98]:
# -----------------------------
# Helper function: estimate particle no-motion steady-state time
# -----------------------------
def estimate_particle_no_motion_time(
    current_time,
    particle_x,
    particle_y,
    particle_z,
    particle_vx,
    particle_vy,
    particle_vz,
    particle_status,
    next_particle_id,
    body_active
):
    """
    Estimate when particle motion will stop.

    Particle no-motion steady state means:
        all particles have been released
        and no active/stuck particles remain

    This estimate is approximate.

    For active particles:
        estimate time to hit floor/outlet using z / downward speed

    For unreleased particles:
        estimate the future release time, then add approximate fall time

    For particles stuck on moving body:
        estimate at least until the body exits, then add approximate fall time
    """

    remaining_times = []

    # -----------------------------
    # Active particles
    # -----------------------------
    active = particle_status == 1

    if np.any(active):
        z_active = particle_z[active]
        vz_active = particle_vz[active]

        downward_speed = np.maximum(-vz_active, particle_speed_floor)

        time_to_floor = z_active / downward_speed

        remaining_times.extend(time_to_floor.tolist())

    # -----------------------------
    # Stuck particles on moving body
    # -----------------------------
    stuck = particle_status == 5

    if np.any(stuck):
        if body_active:
            time_until_body_exits = max(moving_body_exit_time - current_time, 0.0)
        else:
            time_until_body_exits = 0.0

        z_stuck = particle_z[stuck]
        estimated_fall_speed = max(abs(particle_fall_release_vz), particle_speed_floor)

        time_to_fall_after_release = z_stuck / estimated_fall_speed

        remaining_times.extend(
            (time_until_body_exits + time_to_fall_after_release).tolist()
        )

    # -----------------------------
    # Unreleased particles
    # -----------------------------
    unreleased_count = max_particles - next_particle_id

    if unreleased_count > 0:
        # Future releases happen every release_every steps.
        # Estimate the last future release time.
        steps_needed_to_release_all = unreleased_count * release_every
        time_until_last_release = steps_needed_to_release_all * dt

        # Simple fall-time estimate from inlet height.
        estimated_fall_speed = max(U_in, outlet_velocity, particle_speed_floor)
        estimated_fall_time_from_inlet = particle_release_z / estimated_fall_speed

        remaining_times.append(
            time_until_last_release + estimated_fall_time_from_inlet
        )

    # -----------------------------
    # Estimate final no-motion time
    # -----------------------------
    if len(remaining_times) == 0:
        estimated_time = current_time
    else:
        estimated_time = current_time + particle_estimate_safety_factor * max(remaining_times)

    return estimated_time


# -----------------------------
# Reset fields
# -----------------------------
u_field = np.zeros((nz, ny, nx))
v_field = np.zeros((nz, ny, nx))
w_field = np.zeros((nz, ny, nx))
p = np.zeros((nz, ny, nx))

T_field = T_initial * np.ones((nz, ny, nx))
T_field[fixed_obstacle] = T_aluminum

# Initial moving body
body_mask, body_bounds, body_velocity_x, body_active = get_moving_body_mask(0.0)
solid_mask = fixed_obstacle | body_mask
fluid = ~solid_mask

if body_active:
    T_field[body_mask] = T_body

u_field, v_field, w_field = apply_velocity_bcs(
    u_field,
    v_field,
    w_field,
    solid_mask,
    body_mask,
    body_velocity_x
)

T_field = apply_temperature_bcs(
    T_field,
    solid_mask,
    body_mask
)

# -----------------------------
# Particle arrays
# -----------------------------
particle_x = np.full(max_particles, np.nan)
particle_y = np.full(max_particles, np.nan)
particle_z = np.full(max_particles, np.nan)

particle_vx = np.zeros(max_particles)
particle_vy = np.zeros(max_particles)
particle_vz = np.zeros(max_particles)

particle_diameter = np.full(max_particles, np.nan)
particle_tau = np.full(max_particles, np.nan)

particle_body_offset_x = np.full(max_particles, np.nan)
particle_body_offset_y = np.full(max_particles, np.nan)
particle_body_offset_z = np.full(max_particles, np.nan)

# 0 = unreleased
# 1 = active
# 2 = escaped
# 3 = wall deposited
# 4 = fixed obstacle deposited
# 5 = stuck on moving body
particle_status = np.zeros(max_particles, dtype=int)

next_particle_id = 0

# -----------------------------
# Storage for animation
# -----------------------------
stored_T = []

stored_particle_x = []
stored_particle_y = []
stored_particle_z = []
stored_particle_status = []
stored_particle_diameter = []

stored_body_bounds = []
stored_body_velocity_x = []
stored_body_active = []

# -----------------------------
# Storage for steady-state calculation
# -----------------------------
previous_T_for_steady = T_field.copy()

temperature_steady_counter = 0
particle_steady_counter = 0

temperature_steady_time = None
particle_steady_time = None
overall_steady_time = None

estimated_particle_steady_time = None

temperature_residual_history = []
particle_steady_history = []
particle_estimate_history = []

print("Running 3D CFD + temperature + Lagrangian particle simulation...")

for n in tqdm(range(nt), desc="3D simulation progress", unit="step"):

    current_time = (n + 1) * dt

    # Update moving body
    body_mask, body_bounds, body_velocity_x, body_active = get_moving_body_mask(current_time)
    solid_mask = fixed_obstacle | body_mask
    fluid = ~solid_mask

    # 1. Velocity-pressure update
    u_field, v_field, w_field, p = step_velocity_pressure_3d(
        u_field,
        v_field,
        w_field,
        p,
        solid_mask,
        body_mask,
        body_velocity_x
    )

    # 2. Temperature transport
    T_field = step_temperature_3d(
        T_field,
        u_field,
        v_field,
        w_field,
        solid_mask,
        body_mask
    )

    # 3. Release particles
    if n % release_every == 0:
        (
            particle_x,
            particle_y,
            particle_z,
            particle_vx,
            particle_vy,
            particle_vz,
            particle_diameter,
            particle_tau,
            particle_body_offset_x,
            particle_body_offset_y,
            particle_body_offset_z,
            particle_status,
            next_particle_id
        ) = release_particles_3d(
            particle_x,
            particle_y,
            particle_z,
            particle_vx,
            particle_vy,
            particle_vz,
            particle_diameter,
            particle_tau,
            particle_body_offset_x,
            particle_body_offset_y,
            particle_body_offset_z,
            particle_status,
            next_particle_id
        )

    # 4. Move particles
    (
        particle_x,
        particle_y,
        particle_z,
        particle_vx,
        particle_vy,
        particle_vz,
        particle_diameter,
        particle_tau,
        particle_body_offset_x,
        particle_body_offset_y,
        particle_body_offset_z,
        particle_status
    ) = step_particles_3d(
        particle_x,
        particle_y,
        particle_z,
        particle_vx,
        particle_vy,
        particle_vz,
        particle_diameter,
        particle_tau,
        particle_body_offset_x,
        particle_body_offset_y,
        particle_body_offset_z,
        particle_status,
        u_field,
        v_field,
        w_field,
        body_bounds,
        body_velocity_x,
        body_active
    )

    # 5. Store frames
    if n % frame_interval == 0:
        stored_T.append(T_field.copy())

        stored_particle_x.append(particle_x.copy())
        stored_particle_y.append(particle_y.copy())
        stored_particle_z.append(particle_z.copy())
        stored_particle_status.append(particle_status.copy())
        stored_particle_diameter.append(particle_diameter.copy())

        stored_body_bounds.append(body_bounds)
        stored_body_velocity_x.append(body_velocity_x)
        stored_body_active.append(body_active)

    # 6. Steady-state and estimate check
    if (n + 1) % steady_check_interval == 0:

        check_time_interval = steady_check_interval * dt

        # -----------------------------
        # Condition 1:
        # Temperature steady state, dT/dt ≈ 0
        # -----------------------------
        T_change = np.abs(T_field - previous_T_for_steady)

        max_temperature_change_rate = np.nanmax(
            T_change[fluid]
        ) / check_time_interval

        mean_temperature_change_rate = np.nanmean(
            T_change[fluid]
        ) / check_time_interval

        temperature_residual_history.append(
            (
                current_time,
                max_temperature_change_rate,
                mean_temperature_change_rate
            )
        )

        temperature_is_steady = (
            max_temperature_change_rate < temperature_steady_tolerance
        )

        if temperature_is_steady:
            temperature_steady_counter += 1
        else:
            temperature_steady_counter = 0

        if (
            temperature_steady_time is None
            and temperature_steady_counter >= temperature_steady_required_checks
        ):
            temperature_steady_time = current_time

            tqdm.write(
                f"Temperature steady state detected at "
                f"t = {temperature_steady_time:.3f} s, "
                f"max dT/dt = {max_temperature_change_rate:.3e} K/s"
            )

        previous_T_for_steady = T_field.copy()

        # -----------------------------
        # Condition 2:
        # Particle no-motion steady state
        # -----------------------------
        released_count = np.sum(particle_status != 0)
        active_count = np.sum(particle_status == 1)
        escaped_count = np.sum(particle_status == 2)
        wall_count = np.sum(particle_status == 3)
        fixed_obstacle_count = np.sum(particle_status == 4)
        stuck_body_count = np.sum(particle_status == 5)
        stopped_count = escaped_count + wall_count + fixed_obstacle_count

        all_particles_released = next_particle_id >= max_particles
        no_particles_moving = (
            active_count == 0
            and
            stuck_body_count == 0
        )

        particle_motion_is_steady = (
            all_particles_released
            and
            no_particles_moving
        )

        # -----------------------------
        # Estimate particle no-motion steady-state time
        # -----------------------------
        estimated_particle_steady_time = estimate_particle_no_motion_time(
            current_time=current_time,
            particle_x=particle_x,
            particle_y=particle_y,
            particle_z=particle_z,
            particle_vx=particle_vx,
            particle_vy=particle_vy,
            particle_vz=particle_vz,
            particle_status=particle_status,
            next_particle_id=next_particle_id,
            body_active=body_active
        )

        particle_estimate_history.append(
            (
                current_time,
                estimated_particle_steady_time
            )
        )

        particle_steady_history.append(
            (
                current_time,
                released_count,
                active_count,
                escaped_count,
                wall_count,
                fixed_obstacle_count,
                stuck_body_count,
                particle_motion_is_steady,
                estimated_particle_steady_time
            )
        )

        if particle_motion_is_steady:
            particle_steady_counter += 1
        else:
            particle_steady_counter = 0

        if (
            particle_steady_time is None
            and particle_steady_counter >= particle_steady_required_checks
        ):
            particle_steady_time = current_time

            tqdm.write(
                f"Particle no-motion steady state detected at "
                f"t = {particle_steady_time:.3f} s"
            )

        # -----------------------------
        # Overall steady state
        # -----------------------------
        if (
            overall_steady_time is None
            and temperature_steady_time is not None
            and particle_steady_time is not None
        ):
            overall_steady_time = current_time

            tqdm.write(
                f"Overall steady state reached at "
                f"t = {overall_steady_time:.3f} s: "
                f"dT/dt ≈ 0 and no particle motion remains."
            )

        if (
            stop_when_both_steady
            and overall_steady_time is not None
        ):
            tqdm.write(
                f"Stopping simulation early at "
                f"t = {current_time:.3f} s."
            )
            break

    # 7. Status output
    if n % 100 == 0:
        speed_now = np.sqrt(u_field**2 + v_field**2 + w_field**2)
        max_speed = np.nanmax(speed_now[fluid])

        actual_CFL_x = max_speed * dt / dx
        actual_CFL_y = max_speed * dt / dy
        actual_CFL_z = max_speed * dt / dz

        max_T = np.nanmax(T_field[fluid]) - 273.15
        min_T = np.nanmin(T_field[fluid]) - 273.15

        released_count = np.sum(particle_status != 0)
        active_count = np.sum(particle_status == 1)
        escaped_count = np.sum(particle_status == 2)
        wall_count = np.sum(particle_status == 3)
        fixed_obstacle_count = np.sum(particle_status == 4)
        stuck_body_count = np.sum(particle_status == 5)

        if len(temperature_residual_history) > 0:
            latest_temperature_rate = temperature_residual_history[-1][1]
        else:
            latest_temperature_rate = np.nan

        if estimated_particle_steady_time is None:
            estimate_text = "not available"
        else:
            estimate_text = f"{estimated_particle_steady_time:.3f} s"

        tqdm.write(
            f"step {n:4d}/{nt}, "
            f"time = {n * dt:.3f} s, "
            f"body active = {body_active}, "
            f"body vx = {body_velocity_x:.2f} m/s, "
            f"max speed = {max_speed:.4f} m/s, "
            f"CFL = ({actual_CFL_x:.3f}, {actual_CFL_y:.3f}, {actual_CFL_z:.3f}), "
            f"T = {min_T:.2f} to {max_T:.2f} °C, "
            f"max dT/dt = {latest_temperature_rate:.3e} K/s, "
            f"released = {released_count}, "
            f"active = {active_count}, "
            f"escaped = {escaped_count}, "
            f"wall dep = {wall_count}, "
            f"fixed obs dep = {fixed_obstacle_count}, "
            f"stuck body = {stuck_body_count}, "
            f"est particle steady = {estimate_text}"
        )

    # 8. Stop if unstable
    active_or_stuck_particles = (particle_status == 1) | (particle_status == 5)

    if (
        not np.isfinite(u_field).all()
        or not np.isfinite(v_field).all()
        or not np.isfinite(w_field).all()
        or not np.isfinite(T_field).all()
        or not np.isfinite(particle_x[active_or_stuck_particles]).all()
        or not np.isfinite(particle_y[active_or_stuck_particles]).all()
        or not np.isfinite(particle_z[active_or_stuck_particles]).all()
    ):
        print(f"Simulation stopped early at step {n} due to NaN or infinity.")
        break

print("Simulation finished.")
print()

print("Stored frames:", len(stored_T))
print("Particles released:", np.sum(particle_status != 0))
print("Particles active:", np.sum(particle_status == 1))
print("Particles escaped:", np.sum(particle_status == 2))
print("Particles deposited on wall:", np.sum(particle_status == 3))
print("Particles deposited on fixed obstacle:", np.sum(particle_status == 4))
print("Particles stuck on moving body:", np.sum(particle_status == 5))

released = particle_status != 0

if np.any(released):
    d_released = particle_diameter[released]

    print()
    print("Released particle diameter counts")
    print(f"0.3 µm: {np.sum(np.isclose(d_released, 0.3e-6))}")
    print(f"0.5 µm: {np.sum(np.isclose(d_released, 0.5e-6))}")
    print(f"1.0 µm: {np.sum(np.isclose(d_released, 1.0e-6))}")

print()
print("Steady-state results")

if temperature_steady_time is not None:
    print(f"Temperature steady-state time: {temperature_steady_time:.3f} s")
else:
    print("Temperature steady state was not reached within this simulation time.")

if particle_steady_time is not None:
    print(f"Particle no-motion steady-state time: {particle_steady_time:.3f} s")
else:
    print("Particle no-motion steady state was not reached within this simulation time.")

    if estimated_particle_steady_time is not None:
        print(f"Estimated particle no-motion steady-state time: {estimated_particle_steady_time:.3f} s")

        suggested_nt = int(np.ceil(estimated_particle_steady_time / dt))
        suggested_nt_with_margin = int(np.ceil(1.20 * estimated_particle_steady_time / dt))

        print(f"Estimated minimum nt needed: {suggested_nt}")
        print(f"Suggested nt with 20% margin: {suggested_nt_with_margin}")

if overall_steady_time is not None:
    print(f"Overall steady-state time: {overall_steady_time:.3f} s")
else:
    print("Overall steady state was not reached within this simulation time.")
    print("Overall condition: dT/dt ≈ 0 and no particle motion remains.")

if len(temperature_residual_history) > 0:
    final_temperature_rate = temperature_residual_history[-1][1]
    print(f"Final max temperature change rate: {final_temperature_rate:.3e} K/s")

if estimated_particle_steady_time is not None:
    print()
    print("Particle steady-state estimate note:")
    print("This is an approximation based on current active particle positions and speeds.")
    print("Use it to tune nt, but confirm by running the simulation longer.")

Running 3D CFD + temperature + Lagrangian particle simulation...


3D simulation progress:   0%|          | 0/1000 [00:00<?, ?step/s]

step    0/1000, time = 0.000 s, body active = False, body vx = 0.00 m/s, max speed = 2.0000 m/s, CFL = (0.800, 0.800, 0.800), T = 25.00 to 35.00 °C, max dT/dt = nan K/s, released = 1, active = 1, escaped = 0, wall dep = 0, fixed obs dep = 0, stuck body = 0, est particle steady = not available
step  100/1000, time = 4.000 s, body active = True, body vx = 1.50 m/s, max speed = 2.0034 m/s, CFL = (0.801, 0.801, 0.801), T = 25.00 to 36.49 °C, max dT/dt = 1.399e+01 K/s, released = 21, active = 21, escaped = 0, wall dep = 0, fixed obs dep = 0, stuck body = 0, est particle steady = 135.304 s
step  200/1000, time = 8.000 s, body active = False, body vx = 0.00 m/s, max speed = 2.0015 m/s, CFL = (0.801, 0.801, 0.801), T = 25.00 to 35.00 °C, max dT/dt = 2.358e+00 K/s, released = 41, active = 33, escaped = 3, wall dep = 5, fixed obs dep = 0, stuck body = 0, est particle steady = 31152.076 s
step  300/1000, time = 12.000 s, body active = False, body vx = 0.00 m/s, max speed = 2.0015 m/s, CFL = (0.80

## 10. PyVista helper functions

These functions create the 3D boundary geometry:

```text
room box
top rectangular obstacle
bottom rectangular obstacle
circular inlet disk
circular outlet disk
```

They also help plot particle point clouds.

In [99]:
def make_boundary_geometry():
    """
    Create PyVista boundary and fixed obstacle geometry.
    """

    domain_box = pv.Box(bounds=(0, Lx, 0, Ly, 0, Lz))

    top_box = pv.Box(
        bounds=(
            obstacle_x_min,
            obstacle_x_max,
            top_obstacle_y_min,
            top_obstacle_y_max,
            obstacle_z_min,
            obstacle_z_max
        )
    )

    bottom_box = pv.Box(
        bounds=(
            obstacle_x_min,
            obstacle_x_max,
            bottom_obstacle_y_min,
            bottom_obstacle_y_max,
            obstacle_z_min,
            obstacle_z_max
        )
    )

    inlet_disk = pv.Disc(
        center=(inlet_center_x, inlet_center_y, Lz),
        inner=0.0,
        outer=inlet_radius,
        normal=(0.0, 0.0, 1.0),
        r_res=1,
        c_res=64
    )

    outlet_strip = pv.Box(
        bounds=(
            outlet_x_min,
            outlet_x_max,
            outlet_y_min,
            outlet_y_max,
            0.0,
            0.02
        )
    )

    return domain_box, top_box, bottom_box, inlet_disk, outlet_strip


def make_moving_body_geometry(body_bounds):
    """
    Create PyVista geometry for moving body.
    """

    x_min_body, x_max_body, y_min_body, y_max_body, z_min_body, z_max_body = body_bounds

    body_box = pv.Box(
        bounds=(
            x_min_body,
            x_max_body,
            y_min_body,
            y_max_body,
            z_min_body,
            z_max_body
        )
    )

    return body_box


def add_boundary_to_plotter(plotter):
    """
    Add room boundary, fixed floor obstacles, upper inlet, and floor outlet strip.
    """

    domain_box, top_box, bottom_box, inlet_disk, outlet_strip = make_boundary_geometry()

    plotter.add_mesh(
        domain_box,
        style="wireframe",
        color="black",
        line_width=1,
        label="Room boundary"
    )

    plotter.add_mesh(
        top_box,
        color="silver",
        opacity=0.65,
        label="Fixed obstacles"
    )

    plotter.add_mesh(
        bottom_box,
        color="silver",
        opacity=0.65
    )

    plotter.add_mesh(
        inlet_disk,
        color="green",
        opacity=0.8,
        label="Upper inlet"
    )

    plotter.add_mesh(
        outlet_strip,
        color="blue",
        opacity=0.8,
        label="Floor outlet strip"
    )


def add_or_update_moving_body(plotter, body_bounds, body_active):
    """
    Add, replace, or remove moving warm body.
    """

    if "moving_body" in plotter.actors:
        plotter.remove_actor("moving_body")

    if body_active and body_bounds is not None:
        body_box = make_moving_body_geometry(body_bounds)

        plotter.add_mesh(
            body_box,
            name="moving_body",
            color="orange",
            opacity=0.75,
            label="Moving warm body"
        )


def add_points_if_any(plotter, points, name, color, point_size):
    """
    Add or replace particle point cloud.
    """

    if name in plotter.actors:
        plotter.remove_actor(name)

    if points.shape[0] > 0:
        plotter.add_points(
            points,
            name=name,
            color=color,
            point_size=point_size,
            render_points_as_spheres=True
        )

## 11. Animation 1: particle movement + physical boundary setup

This animation shows only:

```text
room boundary
rectangular obstacles
inlet disk
outlet disk
active particles
deposited particles
```

It does not show the temperature field.

In [100]:
save_folder = Path(r"C:\Users\brian\OneDrive\桌面\ASE\cleanroom\animation")
save_folder.mkdir(parents=True, exist_ok=True)

particle_gif_path = save_folder / "3d_particles_boundary_only.gif"

plotter = pv.Plotter(
    off_screen=True,
    window_size=(1000, 800)
)

add_boundary_to_plotter(plotter)

plotter.add_axes()
plotter.add_legend()
plotter.camera_position = "iso"
plotter.camera.zoom(1.15)

plotter.open_gif(str(particle_gif_path), fps=12)

for frame in tqdm(range(len(stored_particle_x)), desc="Saving 3D particle GIF", unit="frame"):

    px = stored_particle_x[frame]
    py = stored_particle_y[frame]
    pz = stored_particle_z[frame]
    ps = stored_particle_status[frame]

    body_bounds = stored_body_bounds[frame]
    body_active = stored_body_active[frame]

    add_or_update_moving_body(plotter, body_bounds, body_active)

    active = ps == 1
    wall_dep = ps == 3
    fixed_obs_dep = ps == 4
    stuck_body = ps == 5

    active_points = np.column_stack((px[active], py[active], pz[active]))
    wall_points = np.column_stack((px[wall_dep], py[wall_dep], pz[wall_dep]))
    fixed_obstacle_points = np.column_stack((px[fixed_obs_dep], py[fixed_obs_dep], pz[fixed_obs_dep]))
    stuck_body_points = np.column_stack((px[stuck_body], py[stuck_body], pz[stuck_body]))

    # Visual marker sizes are unchanged.
    add_points_if_any(
        plotter,
        active_points,
        "active_particles",
        "yellow",
        8
    )

    add_points_if_any(
        plotter,
        wall_points,
        "wall_deposited_particles",
        "red",
        10
    )

    add_points_if_any(
        plotter,
        fixed_obstacle_points,
        "fixed_obstacle_deposited_particles",
        "red",
        10
    )

    add_points_if_any(
        plotter,
        stuck_body_points,
        "stuck_on_moving_body_particles",
        "purple",
        8
    )

    plotter.add_text(
        f"3D particles + one-pass moving body | frame {frame + 1}/{len(stored_particle_x)}",
        name="frame_text",
        position="upper_left",
        font_size=10
    )

    plotter.write_frame()

plotter.close()

print(f"Saved particle animation to: {particle_gif_path.resolve()}")

Saving 3D particle GIF:   0%|          | 0/67 [00:00<?, ?frame/s]

Saved particle animation to: C:\Users\brian\OneDrive\桌面\ASE\cleanroom\animation\3d_particles_boundary_only.gif


## 12. Animation 2: temperature field + physical boundary setup

This animation shows:

```text
room boundary
floor-attached rectangular obstacles
upper inlet disk
lower outlet strip
temperature field slice
```

It does not show particles.

Because the main flow direction is now from upper z face to lower z face, this animation uses a vertical center slice:

```text
y = Ly / 2
```

This makes it easier to see temperature moving from top to bottom.

In [101]:
temperature_gif_path = save_folder / "3d_temperature_boundary_only.gif"

# PyVista ImageData grid
grid = pv.ImageData(
    dimensions=(nx, ny, nz),
    spacing=(dx, dy, dz),
    origin=(0.0, 0.0, 0.0)
)

# Temperature color limits
fixed_fluid_mask = ~fixed_obstacle

all_min = min(float(np.nanmin(T[fixed_fluid_mask] - 273.15)) for T in stored_T)
all_max = max(float(np.nanmax(T[fixed_fluid_mask] - 273.15)) for T in stored_T)

plotter = pv.Plotter(
    off_screen=True,
    window_size=(1000, 800)
)

add_boundary_to_plotter(plotter)

plotter.add_axes()
plotter.add_legend()
plotter.camera_position = "iso"
plotter.camera.zoom(1.15)

plotter.open_gif(str(temperature_gif_path), fps=12)

for frame in tqdm(range(len(stored_T)), desc="Saving 3D temperature GIF", unit="frame"):

    T_c = stored_T[frame].copy() - 273.15
    T_c[fixed_obstacle] = np.nan

    body_bounds = stored_body_bounds[frame]
    body_active = stored_body_active[frame]

    add_or_update_moving_body(plotter, body_bounds, body_active)

    # Convert from [z, y, x] to [x, y, z] for PyVista/VTK
    T_xyz = np.transpose(T_c, (2, 1, 0))

    grid.point_data["Temperature_C"] = T_xyz.ravel(order="F")

    # Vertical middle slice at y = Ly / 2
    temp_slice = grid.slice(
        normal="y",
        origin=(0.0, Ly / 2.0, 0.0)
    )

    plotter.add_mesh(
        temp_slice,
        name="temperature_slice",
        scalars="Temperature_C",
        cmap="coolwarm",
        clim=(all_min, all_max),
        opacity=0.9,
        show_scalar_bar=True,
        scalar_bar_args={"title": "Temperature [C]"}
    )

    plotter.add_text(
        f"3D temperature + one-pass moving body | frame {frame + 1}/{len(stored_T)}",
        name="frame_text",
        position="upper_left",
        font_size=10
    )

    plotter.write_frame()

plotter.close()

print(f"Saved temperature animation to: {temperature_gif_path.resolve()}")

Saving 3D temperature GIF:   0%|          | 0/67 [00:00<?, ?frame/s]

Saved temperature animation to: C:\Users\brian\OneDrive\桌面\ASE\cleanroom\animation\3d_temperature_boundary_only.gif
